In [1]:
import pandas as pd
import numpy as np
from google.colab import files
import io

In [2]:
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
full_off = pd.read_csv(io.BytesIO(uploaded[file_name]), skiprows=1)
full_off.columns = full_off.columns.str.strip()

Saving 2026TeamBatterRanks.csv to 2026TeamBatterRanks.csv


In [3]:
full_off.head()

,#,Team,TG,G,PA,HR,R,RBI,SB,BB%,...,AVG,OBP,SLG,wOBA,xwOBA,wRC+,BsR,Off,Def,WAR
0,1,CHC,162,2379,6382,220,850,804,114,10.6%,...,0.251,0.339,0.428,0.337,0.323,114,3.9,114.0,48.9,37.7
1,2,LAD,162,2407,6214,204,801,756,72,10.1%,...,0.257,0.338,0.424,0.334,0.338,111,-5.6,74.6,-12.5,26.9
2,3,MIL,162,2387,6299,154,832,788,162,10.6%,...,0.258,0.341,0.403,0.328,0.321,107,7.8,63.0,-11.4,26.1
3,4,TBR,162,2385,6106,169,736,700,164,8.4%,...,0.258,0.326,0.402,0.320,0.304,104,4.8,30.3,-12.6,23.8
4,5,CHW,162,2481,6187,211,776,740,93,9.5%,...,0.236,0.320,0.406,0.320,0.318,104,-4.1,27.3,-14.1,23.7


In [4]:
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
full_pit = pd.read_csv(io.BytesIO(uploaded[file_name]), skiprows=1)
full_pit.columns = full_pit.columns.str.strip()

Saving 2026TeamPitcherRanks.csv to 2026TeamPitcherRanks.csv


In [5]:
full_pit.head()

,#,Team,TG,W,L,SV,G,GS,IP,K/9,...,BABIP,LOB%,GB%,HR/FB,vFA (pi),ERA,xERA,FIP,xFIP,WAR
0,1,LAD,162,100,62,46,700,162,1435.2,9.53,...,0.268,75.5%,42.2%,11.2%,95.4,3.55,3.54,3.68,3.74,26.3
1,2,PHI,162,88,74,42,692,162,1438.1,9.87,...,0.309,73.9%,44.8%,13.5%,95.7,4.04,3.61,3.79,3.55,23.7
2,3,BOS,162,87,75,46,650,162,1440.0,8.74,...,0.293,75.6%,43.4%,10.4%,94.2,3.54,3.80,3.65,3.82,23.3
3,4,MIL,162,103,59,55,671,162,1446.2,9.61,...,0.275,75.3%,40.7%,11.3%,95.9,3.51,3.60,3.71,3.76,22.6
4,5,NYY,161,93,68,45,693,161,1441.1,9.05,...,0.272,75.6%,44.2%,11.2%,95.3,3.23,3.69,3.71,3.77,21.9


In [6]:
off = full_off[['Team', 'BB%', 'K%', 'OBP', 'wOBA', 'xwOBA', 'wRC+', 'BsR', 'Def']]

In [7]:
off.head()

,Team,BB%,K%,OBP,wOBA,xwOBA,wRC+,BsR,Def
0,CHC,10.6%,21.1%,0.339,0.337,0.323,114,3.9,48.9
1,LAD,10.1%,20.4%,0.338,0.334,0.338,111,-5.6,-12.5
2,MIL,10.6%,22.0%,0.341,0.328,0.321,107,7.8,-11.4
3,TBR,8.4%,18.8%,0.326,0.320,0.304,104,4.8,-12.6
4,CHW,9.5%,24.1%,0.320,0.320,0.318,104,-4.1,-14.1


In [8]:
pit = full_pit[['Team', 'K/9', 'BB/9', 'GB%' ,'ERA', 'FIP', 'xFIP']]

In [9]:
pit.head()

,Team,K/9,BB/9,GB%,ERA,FIP,xFIP
0,LAD,9.53,3.01,42.2%,3.55,3.68,3.74
1,PHI,9.87,3.03,44.8%,4.04,3.79,3.55
2,BOS,8.74,2.93,43.4%,3.54,3.65,3.82
3,MIL,9.61,3.22,40.7%,3.51,3.71,3.76
4,NYY,9.05,3.05,44.2%,3.23,3.71,3.77


In [10]:
#merge datasets
stats_all = pd.merge(off, pit, on='Team', suffixes=('_off', '_pit'))

# convert percentages to floats
for col in ['BB%', 'K%', 'GB%']:
    if col in stats_all.columns and stats_all[col].dtype == 'object':
        stats_all[col] = stats_all[col].str.rstrip('%').astype('float') / 100.0

# say which stats higher or lower is better
stat_directions = {
    'BB%': True,'K%': False, 'OBP': True, 'wOBA': True, 'xwOBA': True,
    'wRC+': True, 'BsR': True, 'Def': True, 'K/9': True, 'BB/9': False,
    'GB%': True, 'ERA': False, 'FIP': False, 'xFIP': False
}

# normalize each metric
normalized_stats = pd.DataFrame({'Team': stats_all['Team']})

for col, higher_is_better in stat_directions.items():
    if col in stats_all.columns:
        col_min = stats_all[col].min()
        col_max = stats_all[col].max()

        # don't divide by 0
        if col_max - col_min > 0:
            if higher_is_better:
                normalized_stats[col + '_norm'] = (stats_all[col] - col_min) / (col_max - col_min)
            else:
                normalized_stats[col + '_norm'] = (col_max - stats_all[col]) / (col_max - col_min)
        else:
            normalized_stats[col + '_norm'] = 1.0

# 4. take average of all normalized stats as overall_rating
norm_cols = [c for c in normalized_stats.columns if c.endswith('_norm')]
normalized_stats['overall_rating'] = normalized_stats[norm_cols].mean(axis=1) * 100

# display results
display(normalized_stats[['Team', 'overall_rating']].sort_values(by='overall_rating', ascending=False).head())

,Team,overall_rating
1,LAD,77.525610
2,MIL,73.366481
0,CHC,69.162781
6,NYY,59.802130
25,PHI,59.136529


In [11]:
normalized_stats[['Team', 'overall_rating']].sort_values(by='overall_rating', ascending=False)

,Team,overall_rating
1,LAD,77.525610
2,MIL,73.366481
0,CHC,69.162781
6,NYY,59.802130
25,PHI,59.136529
12,BOS,56.895081
3,TBR,56.874530
15,CLE,54.553446
5,SDP,53.661316
19,PIT,53.343986


In [16]:
# Map overall ratings to a dictionary for quick lookup
team_ratings = dict(zip(normalized_stats['Team'], normalized_stats['overall_rating']))

def predict_matchup(team_a, team_b, num_of_games, scale=30):
    """
    Calculates the winning expectancy percentage between team_a and team_b
    using a logistic Bradley-Terry model based on overall_rating.
    """
    # Clean team inputs
    team_a = team_a.strip().upper()
    team_b = team_b.strip().upper()

    if team_a not in team_ratings:
        return f"Error: Team '{team_a}' not found. Please choose from: {', '.join(sorted(team_ratings.keys()))}"
    if team_b not in team_ratings:
        return f"Error: Team '{team_b}' not found. Please choose from: {', '.join(sorted(team_ratings.keys()))}"

    rating_a = team_ratings[team_a]
    rating_b = team_ratings[team_b]

    # Bradley-Terry logit formula
    prob_a = 1 / (1 + 10 ** ((rating_b - rating_a) / scale))
    prob_b = 1 - prob_a

    print(f"=== Matchup: {team_a} vs {team_b} ===")
    print(f"{team_a} Overall Rating: {rating_a:.2f}")
    print(f"{team_b} Overall Rating: {rating_b:.2f}\n")
    print(f"Win Probability / Expectancy:")
    print(f"👉 {team_a}: {prob_a * 100:.2f}%")
    print(f"👉 {team_b}: {prob_b * 100:.2f}%")

    for i in range(num_of_games):
      x = np.random.choice(np.arange(1, 3), p=[prob_a, prob_b])
      if x == 1:
        print(f"Game {i+1}: {team_a} wins!")
      else:
        print(f"Game {i+1}: {team_b} wins!")

In [83]:
# seeing the most lopsided matchup
predict_matchup('cin', 'lad', 7)

=== Matchup: CIN vs LAD ===
CIN Overall Rating: 20.05
LAD Overall Rating: 77.53

Win Probability / Expectancy:
👉 CIN: 1.20%
👉 LAD: 98.80%
Game 1: LAD wins!
Game 2: LAD wins!
Game 3: LAD wins!
Game 4: LAD wins!
Game 5: LAD wins!
Game 6: LAD wins!
Game 7: LAD wins!


In [94]:
# seeing the closest matchup
predict_matchup('tbr', 'bos', 7)

=== Matchup: TBR vs BOS ===
TBR Overall Rating: 56.87
BOS Overall Rating: 56.90

Win Probability / Expectancy:
👉 TBR: 49.96%
👉 BOS: 50.04%
Game 1: TBR wins!
Game 2: BOS wins!
Game 3: TBR wins!
Game 4: TBR wins!
Game 5: BOS wins!
Game 6: TBR wins!
Game 7: BOS wins!


In [90]:
# A function that sees how many simulations it'll take for one team to win an x-game series against another team
# Team A is team you want to win, 'underdog'
def sim_until_series_win(team_a, team_b, num_of_games, scale=30):

    team_a = team_a.strip().upper()
    team_b = team_b.strip().upper()

    rating_a = team_ratings[team_a]
    rating_b = team_ratings[team_b]

    # Bradley-Terry logit formula
    prob_a = 1 / (1 + 10 ** ((rating_b - rating_a) / scale))
    prob_b = 1 - prob_a

    series_count = 0
    won_series = False

    while not won_series:
        series_count += 1
        team_a_wins = 0
        team_b_wins = 0

        # Play up to 7 games until one team wins 4
        for game in range(num_of_games):
            # 1 represents CIN (team_a) win
            winner = np.random.choice([1, 2], p=[prob_a, prob_b])
            if winner == 1:
                team_a_wins += 1
            else:
                team_b_wins += 1

            if team_a_wins == (num_of_games + 1)/2:
                won_series = True
                break
            if team_b_wins == (num_of_games + 1)/2:
                break

    print(f"=== Simulation Results ===")
    print(f"Single game win probability for {team_a}: {prob_a * 100:.4f}%")
    print(f"Number of {team_a} wins: {team_a_wins}")
    print(f"It took {series_count:,} simulated {num_of_games}-game series for {team_a} to defeat {team_b} in a series!")

In [93]:
# simulating the Cardinals' path to a World Series win if they replaced the Phillies as the NL 6 seed and every other outcome remained the same.
sim_until_series_win('stl', 'atl', 3)
print()
sim_until_series_win('stl', 'lad', 5)
print()
sim_until_series_win('stl', 'mil', 7)
print()
sim_until_series_win('stl', 'tbr', 7)

=== Simulation Results ===
Single game win probability for STL: 51.6691%
Number of STL wins: 2
It took 1 simulated 3-game series for STL to defeat ATL in a series!

=== Simulation Results ===
Single game win probability for STL: 7.4224%
Number of STL wins: 3
It took 29 simulated 5-game series for STL to defeat LAD in a series!

=== Simulation Results ===
Single game win probability for STL: 9.9363%
Number of STL wins: 4
It took 239 simulated 7-game series for STL to defeat MIL in a series!

=== Simulation Results ===
Single game win probability for STL: 28.1201%
Number of STL wins: 4
It took 1 simulated 7-game series for STL to defeat TBR in a series!
